# Creating elapsed-time features

Elapsed time answers duration questions: how long between signup and first purchase, how many days since the last contact, or how long an order spent in processing. Durations are often more useful than raw event dates.

## Avoiding leakage

A duration must use only information available at the prediction moment. For example, delivery completion time cannot be used to predict delivery delay before delivery occurs. Define the prediction timestamp first, then calculate durations using events known by that timestamp.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid',context='notebook')

orders=pd.DataFrame({'customer_id':[101,101,101,102,102,103,103],
    'signup_time':['2024-01-02','2024-01-02','2024-01-02','2024-02-10','2024-02-10','2024-03-05','2024-03-05'],
    'order_time':['2024-01-05 10:00','2024-01-20 14:00','2024-02-01 09:00','2024-02-12 08:00','2024-03-01 18:00','2024-03-07 11:00','2024-04-10 16:00'],
    'delivery_time':['2024-01-07 12:00','2024-01-23 14:00','2024-02-05 09:00','2024-02-15 08:00','2024-03-05 18:00','2024-03-10 11:00','2024-04-13 16:00']})
for column in ['signup_time','order_time','delivery_time']:
    orders[column]=pd.to_datetime(orders[column],utc=True)
orders

## 1. Durations between two events

Subtracting two datetime columns returns a Timedelta. Convert it to days or hours with total_seconds. Keep the unit explicit so the feature has a clear interpretation.

In [ ]:
orders['days_to_order']=(orders['order_time']-orders['signup_time']).dt.total_seconds()/86400
orders['hours_to_delivery']=(orders['delivery_time']-orders['order_time']).dt.total_seconds()/3600
display(orders[['customer_id','days_to_order','hours_to_delivery']])

## 2. Time since the previous customer event

Sort by customer and event time before using groupby.shift. The previous order must be calculated within each customer, never across two different customers.

In [ ]:
orders=orders.sort_values(['customer_id','order_time']).copy()
orders['previous_order_time']=orders.groupby('customer_id')['order_time'].shift(1)
orders['days_since_previous_order']=(orders['order_time']-orders['previous_order_time']).dt.total_seconds()/86400
orders['is_first_order']=orders['previous_order_time'].isna()
display(orders[['customer_id','order_time','previous_order_time','days_since_previous_order','is_first_order']])

## 3. Inspect duration distributions

Durations are often skewed: many customers return quickly while a few return much later. A histogram and summary help decide whether to cap, log-transform, bin, or leave the duration unchanged.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,4))
sns.histplot(data=orders,x='days_to_order',bins=10,kde=True,color='#4C78A8',ax=axes[0]); axes[0].set_title('Days from signup to order')
sns.histplot(data=orders,x='days_since_previous_order',bins=10,kde=True,color='#F58518',ax=axes[1]); axes[1].set_title('Days since previous order')
plt.tight_layout(); plt.show()
display(orders[['days_to_order','hours_to_delivery','days_since_previous_order']].describe().T.round(2))

## 4. Practical checks

- Negative durations usually indicate a timestamp or timezone problem.
- Missing previous-event duration often means this is the first event; keep an indicator.
- Use a reference time fixed at prediction time for time-since features.
- For forecasting, split by time and do not allow future events into past rows.
- Validate whether the duration is available at the moment a prediction is requested.

In [ ]:
duration_columns=['days_to_order','hours_to_delivery']
for column in duration_columns:
    assert (orders[column]>=0).all(),f'Negative duration in {column}'
print('Duration checks passed.')
print('First-order rows:',int(orders['is_first_order'].sum()))

## Summary

Elapsed-time features describe how long something took or how long it has been since a known event. They often carry stronger behavioural meaning than raw dates, but only when the event availability and time ordering are respected.